In [ ]:
Implement Matrix Multiplication Using MapReduce

In [ ]:
gedit input.txt

(A 1 1 1
A 1 2 2
A 2 1 3
A 2 2 4
B 1 1 5
B 1 2 6
B 2 1 7
B 2 2 8
)

gedit mapper.py 

(#!/usr/bin/env python3
import sys

# Define matrix dimensions (adjust as per your input)
M = 2  # Rows in A
N = 2  # Columns in A / Rows in B
P = 2  # Columns in B

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    matrix, i, j, value = line.split()
    i, j, value = int(i), int(j), int(value)

    if matrix == 'A':
        # A[i][j] affects all results C[i][k] for k = 1..P
        for k in range(1, P + 1):
            print(f"{i},{k}\tA,{j},{value}")
    else:  # matrix == 'B'
        # B[j][k] affects all results C[i][k] for i = 1..M
        for i_b in range(1, M + 1):
            print(f"{i_b},{j}\tB,{i},{value}")
)

gedit reducer.py
(#!/usr/bin/env python3
import sys
from collections import defaultdict

current_key = None
A_values = defaultdict(int)
B_values = defaultdict(int)

def emit_result(i, k, total):
    print(f"{i},{k}\t{total}")

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    key, value = line.split('\t')
    i, k = map(int, key.split(','))
    matrix, j, val = value.split(',')
    j, val = int(j), int(val)

    if current_key != (i, k):
        # Compute result for previous key
        if current_key is not None:
            total = 0
            for x in range(1, 3):  # j = 1..N
                total += A_values[x] * B_values[x]
            emit_result(current_key[0], current_key[1], total)
        # Reset for new key
        current_key = (i, k)
        A_values.clear()
        B_values.clear()

    # Store partial values
    if matrix == 'A':
        A_values[j] = val
    else:
        B_values[j] = val

# Final key emission
if current_key is not None:
    total = 0
    for x in range(1, 3):
        total += A_values[x] * B_values[x]
    emit_result(current_key[0], current_key[1], total)
)

cat input.txt | python3 mapper.py | sort | python3 reducer.py
